Произведем расчет метрики для бейслайна.
В нашем случае была использована модель llama-3.1-8b-instant с groq.com. У нее большое кол-во токенов и ее потенциально можно запустить локально. Перевод был с en на ru.

Строки уже были переведены заранее, можно свободно запускать расчет метрики.

In [1]:
!pip install groq
!pip install unbabel-comet
import pathlib
from typing import List
from translator_code.db_worker import load_eval_data_for_model
import numpy as np
import pandas as pd

DB_DIR = pathlib.Path("./db")
DB_FILENAME = "parallel_pairs_stream.sqlite"
DB_PATH = DB_DIR / DB_FILENAME

# Ссылка на БД на Google Drive
GDRIVE_URL = "https://drive.google.com/file/d/1rhrsSFS-4KZbK0HYTkmerjXpEsisf7NC/view?usp=share_link"

TARGET_MODEL_NAME = "llama-3.1-8b-instant"

DB_SOURCE_LANG = "en"   # язык исходного текста в формате для бд
DB_TARGET_LANG = "ru"   # язык таргета

COMET_MODEL_NAME = "Unbabel/wmt22-comet-da"

/Users/sergejpolunin/PycharmProjects/StellarisEDA/mod_translation/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sergejpolunin/PycharmProjects/StellarisEDA/mod_translation/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Скачиваем датасет с гугл диска, если он не был скачан для перевода строк.

In [2]:
def ensure_gdown_installed():
    """Устанавливает gdown, если его нет, и возвращает модуль."""
    try:
        import gdown  # noqa: F401
    except ImportError:
        print("gdown не найден")
    finally:
        import gdown
    return gdown


def normalize_gdrive_url(url: str) -> str:
    """
    Принимает ссылку вида
      https://drive.google.com/file/d/<ID>/view?usp=...
    и возвращает
      https://drive.google.com/uc?id=<ID>
    чтобы gdown мог её скачать.
    """
    if "uc?id=" in url:
        return url
    import re
    m = re.search(r"/d/([^/]+)/", url)
    if not m:
        return url
    file_id = m.group(1)
    return f"https://drive.google.com/uc?id={file_id}"


def download_db_if_needed():
    """
    Проверяет наличие БД по пути DB_PATH.
    Если файла нет — создаёт ./db и качает файл с Google Drive.
    """
    if DB_PATH.exists():
        print(f"База уже существует: {DB_PATH}")
        return

    if not GDRIVE_URL:
        raise ValueError("Не задана GDRIVE_URL с ссылкой на БД в Google Drive.")

    DB_DIR.mkdir(parents=True, exist_ok=True)

    gdown = ensure_gdown_installed()
    url = normalize_gdrive_url(GDRIVE_URL)

    print(f"Скачиваю базу из Google Drive в {DB_PATH} ...")
    gdown.download(url, str(DB_PATH), quiet=False)

    if not DB_PATH.exists():
        raise RuntimeError("Не удалось скачать БД с Google Drive.")
    print("База успешно скачана.")

Производим расчет метрики по переведенным строкам.

In [ ]:
from tqdm import tqdm
from translator_code.calc_metric import download_model, load_from_checkpoint, CometScorer, evaluate_translation


def load_comet_scorer(model_name: str) -> CometScorer:
    """Загрузка COMET-модели и обёртки CometScorer."""
    model_path = download_model(model_name)
    comet_model = load_from_checkpoint(model_path)
    return CometScorer(comet_model)


def compute_overall_metric(df: pd.DataFrame, comet_scorer: CometScorer) -> float:
    """
    Считает метрику (final_score) для всех строк и возвращает одно число — среднее.
    COMET считается батчево, теги — построчно.
    """
    if df.empty:
        raise ValueError("В датафрейме нет строк для оценки — возможно, нет данных для этой модели.")

    # 1) Батчом считаем text_score (COMET) для всех строк
    src_list  = df["src"].tolist()
    ref_list  = df["ref"].tolist()
    cand_list = df["cand"].tolist()

    text_scores = comet_scorer.score_batch(
        src_list=src_list,
        ref_list=ref_list,
        cand_list=cand_list,
        batch_size=32,  # можно подправить
    )

    # 2) Считаем теговые метрики/финал с tqdm
    scores: List[float] = []

    for (idx, row), text_score in tqdm(
        zip(df.iterrows(), text_scores),
        total=len(df),
        desc="Eval metric",
        unit="row",
    ):
        result = evaluate_translation(
            ref=row["ref"],
            cand=row["cand"],
            src=row["src"],
            comet_scorer=None,                 # COMET уже посчитан
            precomputed_text_score=text_score, # сюда отдаём precomputed
        )
        scores.append(result["final_score"])

    return float(np.mean(scores))


# Проверяем/скачиваем базу
download_db_if_needed()

# Грузим данные для модели
df_eval = load_eval_data_for_model(
    DB_PATH,
    TARGET_MODEL_NAME,
    src_lang=DB_SOURCE_LANG,
    tgt_lang=DB_TARGET_LANG,
)

print(f"Найдено строк для оценки модели '{TARGET_MODEL_NAME}': {len(df_eval)}")

if df_eval.empty:
    raise SystemExit("Нет данных (строк с ref) для заданной модели — оценивать нечего.")

# Загружаем COMET и считаем метрику
comet_scorer = load_comet_scorer(COMET_MODEL_NAME)
overall_final_score = compute_overall_metric(df_eval, comet_scorer)

print(f"\nИтоговый средний final_score для модели '{TARGET_MODEL_NAME}': {overall_final_score:.4f}")

База уже существует: db/parallel_pairs_stream.sqlite
Найдено строк для оценки модели 'llama-3.1-8b-instant': 7813


Fetching 5 files:  20%|██        | 1/5 [00:00<00:03,  1.01it/s]